# Module 4 - Class 2: Logistic Regression for Churn Classification

**Dataset:** Telco Customer Churn  
**Objective:** Train a logistic regression classifier and understand classification metrics.

### What you will learn
- Logistic regression training and prediction
- Confusion matrix
- Precision, Recall, F1, ROC-AUC
- ROC curve
- Coefficient interpretation (odds ratios)
- Decision threshold tuning

---

## 0. Setup

In [ ]:
# Core data manipulation and visualization libraries
import pandas as pd               # for loading and working with tabular data
import numpy as np                # for numerical operations (e.g., exp for odds ratios)
import matplotlib.pyplot as plt   # for plotting ROC curves, bar charts, etc.
import seaborn as sns             # for the confusion matrix heatmap

# scikit-learn: data splitting
from sklearn.model_selection import train_test_split

# scikit-learn: preprocessing tools
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # scale numeric, encode categorical
from sklearn.compose import ColumnTransformer   # apply different transformations to different columns
from sklearn.pipeline import Pipeline           # chain preprocessing + model into one object
from sklearn.impute import SimpleImputer        # fill in missing values before encoding/scaling

# scikit-learn: the logistic regression classifier
from sklearn.linear_model import LogisticRegression

# scikit-learn: all the classification evaluation metrics we will use
from sklearn.metrics import (
    confusion_matrix,        # 2x2 table of TP, FP, TN, FN
    classification_report,   # formatted precision/recall/F1 per class
    accuracy_score,          # fraction of correct predictions
    precision_score,         # TP / (TP + FP)
    recall_score,            # TP / (TP + FN)
    f1_score,                # harmonic mean of precision and recall
    roc_auc_score,           # area under the ROC curve
    roc_curve                # FPR/TPR pairs at every threshold (for plotting)
)

# Suppress non-critical warnings to keep output clean
import warnings
warnings.filterwarnings('ignore')

print("Setup complete.")

## 1. Load and Preprocess Data

In [ ]:
# Load the Telco Customer Churn dataset directly from GitHub
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# TotalCharges is stored as a string in the raw CSV (some rows have blank spaces instead of numbers)
# errors='coerce' turns those blanks into NaN so we can handle them later with imputation
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Several service columns use 'No internet service' to mean the customer simply doesn't have
# that add-on — standardizing them all to 'No' reduces noise and simplifies encoding
replace_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in replace_cols:
    df[col] = df[col].replace('No internet service', 'No')

# 'No phone service' in MultipleLines means the same thing — collapse it to 'No'
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

# Convert the target from string ('Yes'/'No') to binary integer (1/0)
# 1 = churned, 0 = stayed — required for sklearn classifiers
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Separate features (X) from the target (y)
# Drop customerID because it's just an identifier with no predictive value
X = df.drop(['customerID', 'Churn'], axis=1)
y = df['Churn']

# Identify which columns are numeric and which are categorical
# We'll apply different preprocessing to each group
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

print(f"Shape: {X.shape}")
print(f"Churn rate: {y.mean():.2%}")  # shows class imbalance (typically ~26% churn)

In [ ]:
# ── PREPROCESSING PIPELINE ────────────────────────────────────────────────────

# Numeric pipeline:
# 1. Impute missing values with the column median (robust to outliers vs mean)
# 2. Scale to zero mean and unit variance — logistic regression is sensitive to feature scale
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline:
# 1. Impute missing values with the most frequent category (mode)
# 2. One-hot encode — converts each category into a separate binary column
#    handle_unknown='ignore' silently skips unseen categories at test time
#    drop='first' removes one dummy per feature to avoid multicollinearity (dummy variable trap)
#    sparse_output=False returns a dense NumPy array instead of a sparse matrix
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])

# ColumnTransformer routes each group of columns to its own transformer
# and concatenates the results into a single feature matrix
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, cat_cols)
])

# Split into train (80%) and test (20%)
# stratify=y ensures both splits have the same churn ratio as the full dataset
# This is important because churn is imbalanced (~26% positive class)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. Train Logistic Regression

In [ ]:
# Combine preprocessing and the classifier into a single sklearn Pipeline
# Benefits: no data leakage (fit only on training data), clean single .fit() call,
# and .predict() on raw (unscaled) data just works
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=1000,    # increase from default 100 to ensure the solver converges
        random_state=42   # reproducibility for the solver's random initialization
    ))
])

# Train the full pipeline on the training data
# This fits the preprocessor AND the logistic regression model in one step
pipe.fit(X_train, y_train)

# Hard predictions: 0 (no churn) or 1 (churn) using the default 0.5 threshold
y_pred = pipe.predict(X_test)

# Soft predictions: probability of churn (class 1) for each test sample
# [:, 1] selects the second column — the probability of the positive class
# We need these probabilities for ROC-AUC and threshold tuning later
y_proba = pipe.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

## 3. Confusion Matrix

In [ ]:
# Compute the confusion matrix: rows = actual labels, columns = predicted labels
# Layout: [[TN, FP], [FN, TP]]
cm = confusion_matrix(y_test, y_pred)

# Visualize as a heatmap for easy reading
# annot=True shows the raw counts inside each cell
# fmt='d' formats numbers as integers (not scientific notation)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# Unpack the four values from the flattened confusion matrix
# TN = correct 'no churn' predictions
# FP = predicted churn but actually stayed (false alarm)
# FN = predicted stayed but actually churned (missed churner — often the costly mistake)
# TP = correctly identified churners
tn, fp, fn, tp = cm.ravel()
print(f"True Negatives:  {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives:  {tp}")

## 4. Classification Metrics

In [ ]:
# classification_report gives precision, recall, F1, and support for each class
# 'support' = number of actual samples in that class
# This is more informative than accuracy alone, especially with class imbalance
print("Full Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

# ROC-AUC uses the probability scores (y_proba), not the hard 0/1 predictions
# It measures rank-order quality: how well the model separates churners from non-churners
# across ALL possible thresholds — not just 0.5
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

In [ ]:
# Compute and display each metric individually for easy reference
# Accuracy: overall correct rate — misleading when classes are imbalanced
# Precision: of all predicted churners, what fraction actually churned
# Recall: of all actual churners, what fraction did we catch
# F1: single balanced score when precision and recall trade off against each other
# ROC-AUC: threshold-independent ranking quality
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1': f1_score(y_test, y_pred),
    'ROC-AUC': roc_auc_score(y_test, y_proba)
}

for name, val in metrics.items():
    print(f"{name:>10}: {val:.4f}")

## 5. ROC Curve

In [ ]:
# roc_curve returns FPR and TPR values at every unique probability threshold
# FPR = FP / (FP + TN) — fraction of actual negatives wrongly flagged as positive
# TPR = TP / (TP + FN) — same as Recall — fraction of actual positives correctly caught
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc_val = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(7, 6))

# Plot the model's ROC curve — further toward the top-left corner means better discrimination
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'Logistic Regression (AUC = {auc_val:.3f})')

# Diagonal dashed line = random classifier (AUC = 0.5) — our baseline to beat
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.5)')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. Coefficient Interpretation (Odds Ratios)

In logistic regression, exp(coefficient) gives the odds ratio: how much the odds of churn multiply when the feature increases by 1 unit.

In [ ]:
# Retrieve the fitted OneHotEncoder from inside the pipeline to get the generated column names
# We need these because OHE creates multiple new columns from each original categorical column
ohe = pipe.named_steps['preprocessor'].named_transformers_['cat'].named_steps['encoder']
cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist()

# Reconstruct the full ordered list of feature names after preprocessing
# Order matches: numeric columns first, then all one-hot encoded categorical columns
all_feature_names = numeric_cols + cat_feature_names

# Extract the learned coefficients from the logistic regression model
# coef_[0] is a 1D array because this is a binary (not multi-class) problem
coefs = pipe.named_steps['classifier'].coef_[0]

# Build a summary DataFrame pairing each feature with its coefficient and odds ratio
# Odds ratio = exp(coefficient)
# > 1: the feature increases the odds of churn
# < 1: the feature decreases the odds of churn
# = 1: the feature has no effect
coef_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Coefficient': coefs,
    'Odds Ratio': np.exp(coefs)
}).sort_values('Coefficient', ascending=False)

print("Top 10 features increasing churn:")
print(coef_df.head(10).to_string(index=False))
print()
print("Top 10 features decreasing churn:")
print(coef_df.tail(10).to_string(index=False))

In [ ]:
# Show the top N features visually using a horizontal bar chart
# We take the top half (most positive coefficients) and bottom half (most negative)
# Red bars = increases churn risk; Blue bars = decreases churn risk
top_n = 15
top_features = pd.concat([coef_df.head(top_n // 2 + 1), coef_df.tail(top_n // 2)])

plt.figure(figsize=(10, 6))
colors = ['#d32f2f' if c > 0 else '#1976d2' for c in top_features['Coefficient']]
plt.barh(top_features['Feature'], top_features['Coefficient'], color=colors)
plt.xlabel('Coefficient')
plt.title('Logistic Regression Coefficients (Red = increases churn)')
plt.axvline(x=0, color='black', linewidth=0.8)  # zero line separates positive from negative
plt.tight_layout()
plt.show()

## 7. Threshold Adjustment

By default, logistic regression uses a threshold of 0.5 to classify. Lowering the threshold catches more churners (higher recall) but also more false positives (lower precision).

**Tasks:**
1. Classify using thresholds 0.3, 0.4, and 0.5
2. Compute precision, recall, and F1 for each
3. Build a comparison table
4. Write a brief conclusion: which threshold would you choose for a churn prevention campaign and why?

In [ ]:
# ── THRESHOLD COMPARISON ──────────────────────────────────────────────────────
#
# y_proba contains the model's predicted probability of churn for each customer.
# By default, predict() uses threshold=0.5: if proba >= 0.5 → predict churn.
# Lowering the threshold flags more customers as churners:
#   - Recall goes UP   (we catch more real churners)
#   - Precision goes DOWN (more false alarms among the flagged customers)
#   - F1 may go up or down depending on which effect dominates

thresholds_to_test = [0.3, 0.4, 0.5]
threshold_results = []

for t in thresholds_to_test:
    # Apply the threshold manually: predict churn (1) if probability >= t, else no churn (0)
    y_pred_t = (y_proba >= t).astype(int)

    # Compute the three key metrics for this threshold
    precision = precision_score(y_test, y_pred_t)  # TP / (TP + FP)
    recall    = recall_score(y_test, y_pred_t)     # TP / (TP + FN)
    f1        = f1_score(y_test, y_pred_t)         # 2 * P * R / (P + R)

    # Store results as a dictionary row for the final comparison table
    threshold_results.append({
        'Threshold': t,
        'Precision': round(precision, 4),
        'Recall':    round(recall, 4),
        'F1':        round(f1, 4)
    })

# Convert list of dicts to a DataFrame for a clean side-by-side comparison
threshold_df = pd.DataFrame(threshold_results)
threshold_df = threshold_df.set_index('Threshold')  # use threshold as the row label

print("Threshold Comparison:")
print(threshold_df.to_string())

**Conclusion: Which threshold to choose for a churn prevention campaign?**

For a churn prevention campaign, **threshold = 0.3** is the best choice.

Here is the reasoning:

- In churn prevention, the **cost of a missed churner (False Negative) is much higher** than the cost of a false alarm (False Positive). A missed churner leaves silently and generates zero future revenue. A false alarm just means we send a retention offer to a loyal customer — a small unnecessary cost.

- Lowering the threshold to 0.3 **maximizes Recall** — we catch the largest share of real churners. Even though Precision drops (more false alarms), the business impact of retaining more customers outweighs the cost of extra outreach.

- If the campaign budget is very tight and false alarms are expensive, **threshold = 0.4** is a reasonable compromise — it balances Precision and Recall more evenly and typically gives the best F1 score.

- **Threshold = 0.5** is suitable only when false positives are equally costly as false negatives, which is rarely the case in churn scenarios.

**Rule of thumb:** In churn / fraud / medical diagnosis problems, prefer higher Recall (lower threshold). In spam filtering or loan approval, prefer higher Precision (higher threshold).

---
## Summary

| Concept | Details |
|---------|---------|
| Logistic Regression | Predicts probability of class membership |
| Confusion Matrix | TP, FP, TN, FN breakdown |
| Precision | Of predicted positives, how many are correct |
| Recall | Of actual positives, how many were caught |
| F1 | Harmonic mean of precision and recall |
| ROC-AUC | Area under ROC curve (1.0 = perfect, 0.5 = random) |
| Odds Ratio | exp(coef): multiplicative change in odds per unit increase |
| Threshold | Lower = more recall, less precision |